In [1]:
import sys
import os
import numpy as np
from scipy.spatial.transform import Rotation as R
import argparse
import torch

In [2]:
exec(open('/choreonoid_ws/install/share/irsl_choreonoid/sample/irsl_import.py').read())

In [3]:
# vnoid本家の新しい coordinates ベース実装 (userdir/vnoid, commit abcf1fc "add version
# for using coordinates") を試験導入する。samples/hrp2_vnoid_walking_control.py と同じ
# sys.path 追加パターンで vnoid_python ディレクトリ自体を直接 path に通す
# (new_footstep_planner.py 等が `from vnoid_types import ...` という裸 import をしているため)。
VNOID_PYTHON = os.path.abspath(os.path.join(os.getcwd(), '..', 'userdir', 'vnoid', 'vnoid_python'))
if VNOID_PYTHON not in sys.path:
    sys.path.insert(0, VNOID_PYTHON)

from vnoid_types import Step, Footstep, Param, Ground, Timer, Centroid, Base, Foot
from new_footstep_planner import FootstepPlanner
from new_stepping_controller import SteppingController
from new_stabilizer import Stabilizer
from hrp2_footplane import generate_5step_walking_plan
from irsl_choreonoid.simulation_utils import SimulationEnvironment

In [4]:
dt = 2
render = 0
scene = None
robot = None
viewer = None
time = 0.0

# ロボット固有のパラメータ
joint_names = []
dofs_idx_local = None
num_motors = 0
joint_indices = {}

# 歩行計画
footstep = None
param = None
current_step = 0
step_progress = 0.0  # 0 ~ 1

# SteppingController
stepping_controller = None
stabilizer = None
timer = None
centroid = None
base = None
feet = None

# 内部状態（重心の実際の位置）の追加
com_pos = None

# _apply_full_body_ik()がlegsCOM_IK()に渡す真の重心目標を、既存パイプライン全体が
# 使っている「com_pos ≈ ルートリンク高さ」という慣習から変換するためのオフセット
# (true_CoM - root_pos、startSim()で起立姿勢時に一度だけ実測して設定する)。
com_offset = None

# 目標位置・姿勢
target_joint_angles = None

In [5]:
env_cfg = {
        "num_actions": 30, # ★変更: 全身自由度(12脚+4体幹頭+14腕)
        # joint/link names
        # HRP2_genesis.urdf 用の関節名に変更
        "default_joint_angles": {  # [rad]
            # --- 下半身 (12) ---
            # 膝を伸ばした姿勢（IKが解きやすく、直立高さがbase_init_pos[2]=0.71mに近づく）。
            # 大腿・下腿とも0.30m(URDF実測)なので、脚を完全に伸ばすと股関節高さ≈0.60m+
            # 足底オフセット0.105m≈0.705mとほぼ目標値に一致する。IK特異点回避のため
            # 膝を完全な0ではなく0.30rad(≈17°)だけ残して曲げる。
            "RLEG_JOINT0": 0.0,   # R_HIP_Y
            "RLEG_JOINT1": 0.0,   # R_HIP_R
            "RLEG_JOINT2": -0.15,  # R_HIP_P
            "RLEG_JOINT3": 0.30,   # R_KNEE_P
            "RLEG_JOINT4": -0.15,  # R_ANKLE_P
            "RLEG_JOINT5": 0.0,   # R_ANKLE_R
            "LLEG_JOINT0": 0.0,   # L_HIP_Y
            "LLEG_JOINT1": 0.0,   # L_HIP_R
            "LLEG_JOINT2": -0.15,  # L_HIP_P
            "LLEG_JOINT3": 0.30,   # L_KNEE_P
            "LLEG_JOINT4": -0.15,  # L_ANKLE_P
            "LLEG_JOINT5": 0.0,   # L_ANKLE_R
            
            # --- 体幹・頭部 (4) ---
            "CHEST_JOINT0": 0.0,  # Waist Yaw
            "CHEST_JOINT1": 0.0,  # Waist Pitch
            "HEAD_JOINT0":  0.0,  # Head Yaw
            "HEAD_JOINT1":  0.0,  # Head Pitch
            
            # --- 右腕 (7) ---
            "RARM_JOINT0": 0.0,   # Shoulder P
            "RARM_JOINT1": 0.0,   # Shoulder R
            "RARM_JOINT2": 0.0,   # Shoulder Y
            "RARM_JOINT3": 0.0,   # Elbow P
            "RARM_JOINT4": 0.0,   # Wrist Y
            "RARM_JOINT5": 0.0,   # Wrist P
            "RARM_JOINT6": 0.0,   # Wrist R
            
            # --- 左腕 (7) ---
            "LARM_JOINT0": 0.0,
            "LARM_JOINT1": 0.0,
            "LARM_JOINT2": 0.0,
            "LARM_JOINT3": 0.0,
            "LARM_JOINT4": 0.0,
            "LARM_JOINT5": 0.0,
            "LARM_JOINT6": 0.0,
        },

        "joint_names": [
            # 順序はObservation/Actionと同期します
            "RLEG_JOINT0", "RLEG_JOINT1", "RLEG_JOINT2", "RLEG_JOINT3", "RLEG_JOINT4", "RLEG_JOINT5",
            "LLEG_JOINT0", "LLEG_JOINT1", "LLEG_JOINT2", "LLEG_JOINT3", "LLEG_JOINT4", "LLEG_JOINT5",
            "CHEST_JOINT0", "CHEST_JOINT1",
            "HEAD_JOINT0", "HEAD_JOINT1",
            "RARM_JOINT0", "RARM_JOINT1", "RARM_JOINT2", "RARM_JOINT3", "RARM_JOINT4", "RARM_JOINT5", "RARM_JOINT6",
            "LARM_JOINT0", "LARM_JOINT1", "LARM_JOINT2", "LARM_JOINT3", "LARM_JOINT4", "LARM_JOINT5", "LARM_JOINT6",
        ],
        # PD gains (HRP2は重いので強めに設定)
        "kp": 2000.0,
        "kd": 100.0,
        # termination
        "termination_if_roll_greater_than": 20,  # 転倒判定を少し緩める
        "termination_if_pitch_greater_than": 20,
        # base pose
        # 腰高さの目標値 (ハーフシッティング時)
        "base_init_pos": [0.0, 0.0, 0.71], 
        "base_init_quat": [1.0, 0.0, 0.0, 0.0],
        
        "episode_length_s": 20.0,
        "resampling_time_s": 4.0,
        "action_scale": 0.25, # 学習初期は小さめが安全
        "simulate_action_latency": True,
        "clip_actions": 100.0,
        "dt": 0.01,
        "substeps": 10,
        "rotorInertia": 0.1,
        # noise settings
        "base_roll_noise": [0.00, 0.00],
        "base_pitch_noise": [0.00, 0.00],

        # domain randomization
        "domain_rand": {
            "friction": [0.4, 1.1],     # 地面摩擦係数範囲
            "restitution": [0.0, 0.2],  # 地面反発係数範囲
            "kp": [1800.0, 2200.0],     # Pゲイン範囲 +-10%
            "kd": [80.0, 120.0],        # Dゲイン範囲 +-20%
        },
    }

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
robot_urdf_path = os.path.join(os.getcwd(), "hrp2_description/HRP2_genesis.urdf")

In [7]:
ib.loadRobotItem(cutil.getShareDirectory() + '/model/misc/floor.body')
srobot = RobotModel.loadModelItem(robot_urdf_path, world=True, name='CnoidRobot')

# Separate, purely-kinematic robot model used only to solve IK each control tick
# (see userdir/humanoid_research_k/humanoid_research/walk_sim_vnoid.py::SimMain,
# where a distinct `robot` instance is IK-solved and only its resulting
# angleVector() is pushed to the simulated body via sim.sendAngleVector()).
# world=False keeps it out of the shared WorldItem/AISTSimulatorItem entirely,
# so it is never simulated or collision-checked against the real `srobot` body
# (world=True would place both robots in the same physical world, overlapping
# at the same pose, which is what caused the earlier fall).
ikrobot = RobotModel.loadModelItem(robot_urdf_path, world=False, name='CnoidRobotIK')

# Phase 2: foot force/torque sensors for a real-measured-ZMP stabilizer.
# HRP2_genesis.urdf defines no ForceSensor devices, so add them programmatically
# to srobot's ankle-roll links (RLEG_LINK5/LLEG_LINK5 - the link each ankle-roll
# joint, RLEG_JOINT5/LLEG_JOINT5, drives; the same location used as
# registerEndEffector's tip_link below) at the link's local origin, matching the
# standard Choreonoid convention for declaring a ForceSensor directly on a link
# with no extra local offset (see e.g. share/model/SR1/SR1.body's
# `type: ForceSensor` elements under LLEG_LINK6/RLEG_LINK6). Only srobot (the
# body actually driven by AISTSimulatorItem) needs real sensors - ikrobot is a
# pure kinematic scratch model and is never simulated.
import cnoid.Body as cnoidBody
foot_force_sensors = {}
for side, link_name in (('lleg', 'LLEG_LINK5'), ('rleg', 'RLEG_LINK5')):
    fs = cnoidBody.ForceSensor()
    fs.setName(side + '_force_sensor')
    link = srobot.robot.link(link_name)
    srobot.robot.addDevice(fs, link)
    foot_force_sensors[side] = fs
srobot.robot.initializeDeviceStates()

In [8]:
sim = SimulationEnvironment('CnoidRobot')
sim.stop()

In [9]:
def setrobot():
    for bot in (srobot, ikrobot):
        bot.translate(fv(0, 0, 0.71))
        bot.setAngleMap( env_cfg["default_joint_angles"] )

In [10]:
setrobot()

In [11]:
for idx, lk in enumerate(srobot.robot.links):
    print('link {}: {}'.format(idx, lk.name))
for idx, j in enumerate(srobot.robot.joints):
    print('joint {}: {}'.format(idx, j.jointName))

link 0: WAIST
link 1: RLEG_LINK0
link 2: RLEG_LINK1
link 3: RLEG_LINK2
link 4: RLEG_LINK3
link 5: RLEG_LINK4
link 6: RLEG_LINK5
link 7: LLEG_LINK0
link 8: LLEG_LINK1
link 9: LLEG_LINK2
link 10: LLEG_LINK3
link 11: LLEG_LINK4
link 12: LLEG_LINK5
link 13: CHEST_LINK0
link 14: CHEST_LINK1
link 15: HEAD_LINK0
link 16: HEAD_LINK1
link 17: RARM_LINK0
link 18: RARM_LINK1
link 19: RARM_LINK2
link 20: RARM_LINK3
link 21: RARM_LINK4
link 22: RARM_LINK5
link 23: RARM_LINK6
link 24: LARM_LINK0
link 25: LARM_LINK1
link 26: LARM_LINK2
link 27: LARM_LINK3
link 28: LARM_LINK4
link 29: LARM_LINK5
link 30: LARM_LINK6
joint 0: RLEG_JOINT0
joint 1: RLEG_JOINT1
joint 2: RLEG_JOINT2
joint 3: RLEG_JOINT3
joint 4: RLEG_JOINT4
joint 5: RLEG_JOINT5
joint 6: LLEG_JOINT0
joint 7: LLEG_JOINT1
joint 8: LLEG_JOINT2
joint 9: LLEG_JOINT3
joint 10: LLEG_JOINT4
joint 11: LLEG_JOINT5
joint 12: CHEST_JOINT0
joint 13: CHEST_JOINT1
joint 14: HEAD_JOINT0
joint 15: HEAD_JOINT1
joint 16: RARM_JOINT0
joint 17: RARM_JOINT1
joint

In [12]:
for bot in (srobot, ikrobot):
    bot.registerEndEffector('lleg', ## end-effector
                              'LLEG_JOINT5', ## tip-link
                              tip_link_to_eef=coordinates(fv(0, 0, -0.105)),
                              joint_tuples = (('LLEG_JOINT0', 'hip-y'),
                                              ('LLEG_JOINT1', 'hip-r'),
                                              ('LLEG_JOINT2', 'hip-p'),
                                              ('LLEG_JOINT3', 'knee-p'),
                                              ('LLEG_JOINT4', 'ankle-p'),
                                              ('LLEG_JOINT5', 'ankle-r'),
                                              )
                              )
    bot.registerEndEffector('rleg', ## end-effector
                              'RLEG_JOINT5', ## tip-link
                              tip_link_to_eef=coordinates(fv(0, 0, -0.105)),
                              joint_tuples = (('RLEG_JOINT0', 'hip-y'),
                                              ('RLEG_JOINT1', 'hip-r'),
                                              ('RLEG_JOINT2', 'hip-p'),
                                              ('RLEG_JOINT3', 'knee-p'),
                                              ('RLEG_JOINT4', 'ankle-p'),
                                              ('RLEG_JOINT5', 'ankle-r'),
                                              )
                              )
     

In [13]:
srobot.rleg

In [14]:
srobot.lleg.jointNames

['LLEG_JOINT0',
 'LLEG_JOINT1',
 'LLEG_JOINT2',
 'LLEG_JOINT3',
 'LLEG_JOINT4',
 'LLEG_JOINT5']

In [15]:
srobot.rleg.endEffector

<coordinates[0x5696ad02b040] 1.67921e-17 -0.095 0.0117374 / 0 -1.38778e-17 0 1 >

In [16]:
di = DrawInterface()
cds=mkshapes.makeCoords(coords=srobot.rleg.endEffector, length=0.14, lineWidth=5)
srobot.assoc(cds, srobot.rleg.tipLink)
di.addObject(cds)

In [17]:
### check joint names
setAnglesToCnoid = None
setAnglesToGs = None
jcnoid = srobot.jointNames
jgs = env_cfg["joint_names"]
for a, b in zip(jcnoid, jgs):
    if a != b:
        ## self.env_cfg["joint_names"] -> self.srobot.jointNames
        setAnglesToCnoid = [ jcnoid.index(n) for n in jgs ] # setAnglesoToCnoid タイプミス?
        ## self.srobot.jointNames -> self.env_cfg["joint_names"]
        setAnglesToGs = [ jgs.index(n) for n in jcnoid ] # setAnglesoToGs タイプミス?
        break

In [18]:
def _build_walking_plan(stride=0.10, spacing=0.20, com_height=0.71, step_duration=0.80, time_constant=0.40, num_steps=5):
    param = Param(com_height=com_height, T=time_constant)
    steps = []

    left_x = 0.0
    right_x = 0.0
    left_y = spacing * 0.5
    right_y = -spacing * 0.5

    for index in range(num_steps):
        step = Step()
        step.side = 0 if index % 2 == 0 else 1
        step.duration = step_duration
        step.stride = stride
        step.spacing = spacing
        step.turn = 0.0
        step.sway = 0.0
        step.climb = 0.0
        step.stepping = index != 0

        if index == 0:
            left_x = 0.0
            right_x = 0.0
        elif index % 2 == 1:
            right_x += stride
        else:
            left_x += stride

        step.foot_coords[0] = coordinates(fv(left_x, left_y, 0.0))
        step.foot_coords[1] = coordinates(fv(right_x, right_y, 0.0))
        step.zmp = (step.foot_coords[0].pos + step.foot_coords[1].pos) * 0.5
        step.dcm = step.zmp + np.array([0.0, 0.0, com_height], dtype=np.float64)
        steps.append(step)

    footstep = Footstep(steps=steps)
    FootstepPlanner().GenerateDCM(param, footstep)
    footstep.steps[0].stepping = False
    return footstep, param


def _foot_to_coords(foot, ankle_height_offset=0.0):
    # The registered end-effector (see registerEndEffector cell) is already
    # tip_link_to_eef=(0,0,-0.105), matching the URDF's exact ankle-joint-to-sole
    # distance - so the IK target here should be the sole position (z=0 for
    # ground contact) directly, with no additional offset. ankle_height_offset
    # is kept as an optional hook (e.g. to command the foot above the ground)
    # but defaults to 0 to avoid double-counting the ankle/sole offset.
    tgt = foot.coords_ref.copy()
    if ankle_height_offset:
        tgt.pos = tgt.pos + fv(0, 0, ankle_height_offset)
    return tgt


def _apply_ik_targets():
    try:
        srobot.lleg.inverseKinematics(_foot_to_coords(feet[0]))
    except Exception:
        pass

    try:
        srobot.rleg.inverseKinematics(_foot_to_coords(feet[1]))
    except Exception:
        pass


def startSim():
    global footstep, footstep_buffer, param, stepping_controller, timer, centroid, base, feet, com_pos, current_step, step_progress, target_joint_angles, init_cds

    sim.stop()
    sim.start(dt=env_cfg["dt"] / env_cfg["substeps"],
              P=env_cfg["kp"], D=env_cfg["kd"],
              simple=True,
              effortRange=env_cfg['effortRange'] if 'effortRange' in env_cfg else True,
              rotorInertia=env_cfg['rotorInertia'] if 'rotorInertia' in env_cfg else None)
    init_cds = sim.sbody.rootLink.getCoords()

    footstep, param = _build_walking_plan()
    footstep_buffer = Footstep(steps=[Step(), Step()])
    stepping_controller = SteppingController()
    stepping_controller.debug = 0
    timer = Timer()
    timer.dt = env_cfg["dt"] / env_cfg["substeps"]
    centroid = Centroid()
    base = Base()
    feet = [Foot(), Foot()]

    st0 = footstep.steps[0]
    feet[0].coords_ref = st0.foot_coords[0].copy()
    feet[1].coords_ref = st0.foot_coords[1].copy()
    feet[0].contact_ref = True
    feet[1].contact_ref = True

    centroid.zmp_ref = st0.zmp.copy()
    centroid.zmp_target = st0.zmp.copy()
    centroid.dcm_ref = st0.dcm.copy()
    centroid.dcm_target = st0.dcm.copy()
    centroid.com_pos_ref = st0.dcm.copy()
    com_pos = st0.dcm.copy()

    current_step = 0
    step_progress = 0.0
    target_joint_angles = None

In [19]:
def _build_walking_plan(stride=0.10, spacing=0.20, com_height=0.71, step_duration=0.80, time_constant=0.40, num_steps=10):
    param = Param(com_height=com_height, T=time_constant)
    steps = []

    left_x = 0.0
    right_x = 0.0
    left_y = spacing * 0.5
    right_y = -spacing * 0.5

    for index in range(num_steps):
        step = Step()
        step.side = 0 if index % 2 == 0 else 1
        step.duration = step_duration
        step.stride = stride
        step.spacing = spacing
        step.turn = 0.0
        step.sway = 0.0
        step.climb = 0.0
        step.stepping = index != 0

        if index == 0:
            left_x = 0.0
            right_x = 0.0
        elif index % 2 == 1:
            right_x += stride
        else:
            left_x += stride

        step.foot_coords[0] = coordinates(fv(left_x, left_y, 0.0))
        step.foot_coords[1] = coordinates(fv(right_x, right_y, 0.0))
        step.zmp = (step.foot_coords[0].pos + step.foot_coords[1].pos) * 0.5
        step.dcm = step.zmp + np.array([0.0, 0.0, com_height], dtype=np.float64)
        steps.append(step)

    footstep = Footstep(steps=steps)
    FootstepPlanner().GenerateDCM(param, footstep)
    footstep.steps[0].stepping = False
    return footstep, param



def _foot_to_coords(foot, ankle_height_offset=0.0):
    # The registered end-effector (see registerEndEffector cell) is already
    # tip_link_to_eef=(0,0,-0.105), matching the URDF's exact ankle-joint-to-sole
    # distance - so the IK target here should be the sole position (z=0 for
    # ground contact) directly, with no additional offset. ankle_height_offset
    # is kept as an optional hook (e.g. to command the foot above the ground)
    # but defaults to 0 to avoid double-counting the ankle/sole offset.
    #
    # Phase 2: if _advance_force_tracking() (see USE_FORCE_TRACKING below) has
    # computed a compliance-corrected target this tick, use it instead of the
    # raw plan target foot.coords_ref. foot._compliance_coords is a dynamic
    # attribute (like foot.force/moment) set fresh each tick by
    # _advance_force_tracking() - falling back to coords_ref keeps this
    # function's behavior byte-for-byte unchanged when force tracking is off.
    tgt = getattr(foot, '_compliance_coords', None)
    if tgt is None:
        tgt = foot.coords_ref
    tgt = tgt.copy()
    if ankle_height_offset:
        tgt.pos = tgt.pos + fv(0, 0, ankle_height_offset)
    return tgt



def _ground_foot_target():
    """Target coordinates that place the foot midpoint on the ground plane."""
    return coordinates(fv(0.0, 0.0, 0.0))



def _pose_standing(bot):
    try:
        bot.fixLegToCoords(_ground_foot_target())
        bot.moveCentroidOnFoot(com_height=env_cfg["base_init_pos"][2])
        bot.flush()
    except Exception:
        try:
            bot.rootCoords(coordinates(fv(0.0, 0.0, env_cfg["base_init_pos"][2])))
            bot.flush()
        except Exception:
            pass



def startSim(settle_duration=5.0):
    global footstep, footstep_buffer, param, stepping_controller, stabilizer, timer, centroid, base, feet, com_pos, com_offset, current_step, step_progress, target_joint_angles, init_cds, time

    sim.stop()

    # Earlier interactive cells in this notebook (IK demo cells further above)
    # may have left srobot/ikrobot's joints in a partial/failed-IK configuration
    # (e.g. a leg bent from a move() whose matching inverseKinematics() didn't
    # converge). Reset both to the clean default pose before computing the
    # standing configuration, so startSim() never depends on what interactive
    # cells happened to run earlier in the notebook.
    for bot in (srobot, ikrobot):
        bot.setAngleMap(env_cfg["default_joint_angles"])

    # Pose the kinematic reference models into a standing configuration *before*
    # starting the physics simulation, then explicitly store srobot's pose as the
    # item's initial state (matching irsl_rl/rl_env_cnoid.py::reset_env_idx() and
    # userdir/humanoid_research_k/humanoid_research/walk_sim.py::startSim()).
    # Without storeInitialState(), sim.start()/startSimulation() resets the
    # simulated body back to the URDF's original (unposed, all-zero-angle) state,
    # regardless of how srobot was kinematically posed beforehand.
    _pose_standing(srobot)
    srobot.item.storeInitialState()
    _pose_standing(ikrobot)

    # legsCOM_IK() (see _apply_full_body_ik()) drives the *true* mass-weighted
    # center of mass, not the root-link position - but the rest of this
    # pipeline (DCM/ZMP planning, com_pos integration) treats "com_pos" as
    # root-link height (it grew out of the old per-leg-IK + rootCoords()
    # approach). Measured at this standing pose: root_z ~= 0.53m but true
    # CoM_z ~= 0.71m (HRP2's arms/chest/head pull the real CoM well above the
    # waist). Without this offset, legsCOM_IK() was commanding the real CoM
    # down to ~0.53m, forcing a deep, unstable crouch - which is what caused
    # Phase 1's initial regression (falling within ~60 ticks, before this fix).
    com_offset = np.array(ikrobot.centerOfMass, dtype=np.float64) - np.array(ikrobot.rootCoords().pos, dtype=np.float64)

    sim.start(dt=env_cfg["dt"] / env_cfg["substeps"],
              P=env_cfg["kp"], D=env_cfg["kd"],
              simple=True,
              effortRange=env_cfg['effortRange'] if 'effortRange' in env_cfg else True,
              rotorInertia=env_cfg['rotorInertia'] if 'rotorInertia' in env_cfg else None)
    init_cds = sim.sbody.rootLink.getCoords()

    # Let contact/interpenetration from spawn settle out before any footstep-driven
    # CoM motion begins. No new target is queued here, so the PD controller simply
    # holds the stored initial pose while gravity/contacts settle the body.
    # Same idea as irsl_choreonoid.simulation_utils.startSimEnv()'s pre_wait.
    if settle_duration > 0.0:
        sim.run(settle_duration, update=None, stop=False)

    # _pose_standing()'s moveCentroidOnFoot(com_height=env_cfg["base_init_pos"][2])
    # targets a fixed 0.71m, but IK/gravity/PD don't necessarily converge to
    # exactly that height (env_cfg's default_joint_angles keeps the knee bent by
    # 0.30rad to avoid an IK singularity, so the actual standing height held
    # throughout the settle period above is usually somewhat below 0.71m). Read
    # back what height was *actually* held and use that as the walking plan's
    # com_height, instead of the fixed 0.71 default - otherwise com_pos snaps
    # from the real settled height to 0.71m the instant the walking loop starts,
    # which yanks the knee straight (the "recoil" visible right at t=5s).
    actual_root_z = sim.sbody.rootLink.getCoords().pos[2]

    time = 0.0

    footstep, param = _build_walking_plan(stride=0.08, spacing=0.25, step_duration=1.0, num_steps=1000, com_height=actual_root_z)
    # Physical parameters used only by the base-tilt feedback stabilizer path
    # (see _advance_com_state_stabilized() / USE_STABILIZER_FEEDBACK below).
    # total_mass is read from the actual loaded robot model rather than the
    # Param default placeholder; nominal_inertia/zmp_min/zmp_max keep the
    # Param dataclass's approximate defaults (see vnoid_types.Param).
    param.total_mass = srobot.mass
    footstep_buffer = Footstep(steps=[Step(), Step()])
    stepping_controller = SteppingController()
    # new_stepping_controller.SteppingController defaults to debug=4 (very verbose
    # per-tick internal-state dumps, unlike the old array-based version which
    # defaulted to no debug output). Match walking_control.py's own convention
    # (`self.wc.stepping_controller.debug = 0`) to keep the loop's actual output
    # readable and avoid a huge/slow notebook output over hundreds of ticks.
    stepping_controller.debug = 0
    # Default swing_height (foot lift height during the swing phase) is 0.05m
    # (see new_stepping_controller.SteppingController.__init__). Testing a higher
    # value here to see whether more foot clearance reduces scuffing/tripping
    # against the ground during swing and delays falls.
    stepping_controller.swing_height = 0.10
    stabilizer = Stabilizer()
    # Phase 2: force/moment tracking compliance gains (see
    # _advance_force_tracking() / FORCE_CTRL_SCALE below). Stabilizer.__init__'s
    # reference defaults are all exactly 0 (force_ctrl_gain/moment_ctrl_gain/
    # force_ctrl_limit/moment_ctrl_limit), which makes this control path a
    # total no-op regardless of any scale factor - there is no "reference"
    # value to fall back on, so these are placeholder starting points (same
    # spirit as Param.nominal_inertia's placeholder in vnoid_types.py) meant to
    # be swept/tuned via FORCE_CTRL_SCALE, not treated as known-good. force_ctrl
    # gain has units of 1/N*(m/s^2)-ish (position correction rate per unit force
    # error) and moment_ctrl_gain similarly for orientation per unit moment
    # error; limits are in meters/radians for the integrated correction.
    #
    # FORCE_GAIN_MULTIPLIER (see that cell) scales these two base gains by
    # decades to find where an effect emerges at all - force_ctrl_limit/
    # moment_ctrl_limit stay fixed as a safety envelope on the integrated
    # correction regardless of gain magnitude.
    stabilizer.force_ctrl_gain = 1.0e-5 * FORCE_GAIN_MULTIPLIER
    stabilizer.moment_ctrl_gain = 1.0e-5 * FORCE_GAIN_MULTIPLIER
    stabilizer.force_ctrl_damping = 1.0
    stabilizer.moment_ctrl_damping = 1.0
    stabilizer.force_ctrl_limit = 0.03
    stabilizer.moment_ctrl_limit = 0.1
    timer = Timer()
    timer.dt = env_cfg["dt"]
    centroid = Centroid()
    base = Base()
    # angvel_ref defaults to None in vnoid_types.Base; the base-tilt feedback
    # path (Stabilizer.CalcBaseTilt) integrates this every tick, so it needs a
    # real zero vector to start from.
    base.angvel_ref = np.zeros(3, dtype=np.float64)
    feet = [Foot(), Foot()]

    st0 = footstep.steps[0]
    feet[0].coords_ref = st0.foot_coords[0].copy()
    feet[1].coords_ref = st0.foot_coords[1].copy()
    feet[0].contact_ref = True
    feet[1].contact_ref = True

    centroid.zmp_ref = st0.zmp.copy()
    centroid.zmp_target = st0.zmp.copy()
    centroid.dcm_ref = st0.dcm.copy()
    centroid.dcm_target = st0.dcm.copy()
    centroid.com_pos_ref = st0.dcm.copy()
    com_pos = st0.dcm.copy()

    current_step = 0
    step_progress = 0.0
    target_joint_angles = None

In [20]:
sim.stop()

In [21]:

for idx, lk in enumerate(srobot.robot.links):
    print('link {}: {}'.format(idx, lk.name))
for idx, j in enumerate(srobot.robot.joints):
    print('joint {}: {}'.format(idx, j.jointName))

link 0: WAIST
link 1: RLEG_LINK0
link 2: RLEG_LINK1
link 3: RLEG_LINK2
link 4: RLEG_LINK3
link 5: RLEG_LINK4
link 6: RLEG_LINK5
link 7: LLEG_LINK0
link 8: LLEG_LINK1
link 9: LLEG_LINK2
link 10: LLEG_LINK3
link 11: LLEG_LINK4
link 12: LLEG_LINK5
link 13: CHEST_LINK0
link 14: CHEST_LINK1
link 15: HEAD_LINK0
link 16: HEAD_LINK1
link 17: RARM_LINK0
link 18: RARM_LINK1
link 19: RARM_LINK2
link 20: RARM_LINK3
link 21: RARM_LINK4
link 22: RARM_LINK5
link 23: RARM_LINK6
link 24: LARM_LINK0
link 25: LARM_LINK1
link 26: LARM_LINK2
link 27: LARM_LINK3
link 28: LARM_LINK4
link 29: LARM_LINK5
link 30: LARM_LINK6
joint 0: RLEG_JOINT0
joint 1: RLEG_JOINT1
joint 2: RLEG_JOINT2
joint 3: RLEG_JOINT3
joint 4: RLEG_JOINT4
joint 5: RLEG_JOINT5
joint 6: LLEG_JOINT0
joint 7: LLEG_JOINT1
joint 8: LLEG_JOINT2
joint 9: LLEG_JOINT3
joint 10: LLEG_JOINT4
joint 11: LLEG_JOINT5
joint 12: CHEST_JOINT0
joint 13: CHEST_JOINT1
joint 14: HEAD_JOINT0
joint 15: HEAD_JOINT1
joint 16: RARM_JOINT0
joint 17: RARM_JOINT1
joint

In [22]:
srobot.moveCentroidOnFoot() ## 重心を両足の中心にする

(True, 2)

In [23]:
etgt=mkshapes.makeCoords(coords=srobot.rleg.endEffector, length=0.2, lineWidth=3)
di.addObject(etgt)

In [24]:
srobot.rleg.move(fv(0.0, 0.0, 0.06), constraint='xyz')

(True, 10)

In [25]:
srobot.rleg.inverseKinematics(etgt)

(True, 11)

In [26]:
def _sync_robot_state_from_sim():
    """Read the current simulated state when the API exposes it.

    Note: `com_pos` is deliberately NOT synced from the simulated body here.
    Matching vnoid_python's WalkingControl.step_simulation(), `com_pos` (our
    stand-in for centroid.com_pos_ref) is a purely open-loop, plan-driven
    kinematic quantity fed into leg IK; it is only ever advanced by
    _advance_com_state()'s own integration. Overwriting it with the real
    (currently near-stationary) simulated root position every tick discarded
    each tick's forward integration before it could accumulate."""
    root_link = getattr(getattr(sim, 'sbody', None), 'rootLink', None)
    if root_link is None:
        return

    try:
        root_coords = root_link.getCoords()
        if root_coords is not None:
            base.coords = root_coords.copy()
    except Exception:
        pass

    try:
        if hasattr(root_link, 'v'):
            base.vel = np.array(root_link.v, dtype=np.float64)
        elif hasattr(root_link, 'vel'):
            base.vel = np.array(root_link.vel, dtype=np.float64)
    except Exception:
        pass

    try:
        if hasattr(root_link, 'w'):
            base.angvel = np.array(root_link.w, dtype=np.float64)
        elif hasattr(root_link, 'omega'):
            base.angvel = np.array(root_link.omega, dtype=np.float64)
    except Exception:
        pass


# Phase 2: local-frame ankle->sole moment-arm correction, matching the exact
# offset already used everywhere else in this notebook for IK/footstep
# placement (registerEndEffector's tip_link_to_eef=(0,0,-0.105) - see the
# registerEndEffector cell). ForceSensor devices (foot_force_sensors /
# srobot.robot.addDevice(), see the srobot-loading cell) sit at the ankle-roll
# link's origin, but Stabilizer.CalcZmp() assumes force/moment are measured at
# the point whose local ZMP is being computed (the sole) - so the raw sensor
# torque needs shifting from the ankle to the sole: tau_sole = tau_ankle -
# (ankle_to_sole x force), the standard rigid-body wrench transport formula.
_ANKLE_TO_SOLE = fv(0, 0, -0.105)
_FOOT_SENSOR_NAMES = ('lleg_force_sensor', 'rleg_force_sensor')


def _find_device_by_name(body, name):
    """cnoid.Body.Body.device() (the raw native binding, as opposed to
    RobotModelWrapped.device()) only accepts an integer index, not a name
    string - passing a name raises a TypeError. Confirmed via direct probing
    (sim.sbody.devices does contain both ForceSensor devices with the right
    names/types - they clone into the simulated body correctly - the lookup
    method was simply wrong). Search the device list by .name instead."""
    if body is None:
        return None
    for d in body.devices:
        if d.name == name:
            return d
    return None


def _update_foot_wrenches_from_sim():
    """Read real foot force/torque sensor values from the simulated body
    (see foot_force_sensors / srobot.robot.addDevice() setup cell) and correct
    the moment from the ankle-roll joint origin (where the sensor sits) to the
    sole (see _ANKLE_TO_SOLE above). Falls back to zero force/moment if the
    sensor device can't be found (e.g. before sim.start())."""
    sbody = getattr(sim, 'sbody', None)
    for foot, sensor_name in zip(feet, _FOOT_SENSOR_NAMES):
        force = np.zeros(3, dtype=np.float64)
        moment = np.zeros(3, dtype=np.float64)
        fs = _find_device_by_name(sbody, sensor_name)
        if fs is not None:
            force = np.array(fs.f, dtype=np.float64)
            moment = np.array(fs.tau, dtype=np.float64) - np.cross(_ANKLE_TO_SOLE, force)
        foot.force = force
        foot.moment = moment


def _update_measured_zmp():
    """Phase 2, validation stage: compute the measured-ZMP quantities
    (Foot.contact/balance/zmp, Centroid.zmp) via Stabilizer.CalcZmp() from the
    real sensor readings _update_foot_wrenches_from_sim() just populated.
    Deliberately does not yet feed back into com_pos/centroid.zmp_ref/dcm_ref -
    this is purely computed so it can be inspected (see the validation cell)
    before any control loop depends on it, following the staged rollout in the
    approved plan (see plan file: verify raw sensor values are sane before
    wiring in force/moment tracking control)."""
    stabilizer.CalcZmp(param, centroid, feet)


# Toggle for the A/B comparison requested by the user: False reproduces the
# existing pure open-loop baseline (_advance_com_state, matching vnoid's
# Stabilizer.CalcDcmDynamicsSimple()); True routes through
# _advance_com_state_stabilized() instead, which feeds the *measured* base
# roll/pitch and angular velocity (already read every tick by
# _sync_robot_state_from_sim() above) back into the CoM trajectory via
# Stabilizer.CalcBaseTilt()+CalcRecoveryDelta() (ported from
# userdir/vnoid/vnoid_python/stabilizer.py, the reference array-based
# implementation). See run_trial() below for how both are exercised in the
# same notebook run.
USE_STABILIZER_FEEDBACK = False

# Multiplies the recovery-moment disturbance (delta) applied inside
# _advance_com_state_stabilized(); 1.0 = full reference gains, 0.0 = pure
# open-loop (should behave identically to _advance_com_state()). Used to
# sweep the feedback strength while isolating it from the two prior bugs
# (footstep-plan corruption, and forward-Euler integration error) that are
# now fixed - see that function's docstring for the debugging history.
DELTA_SCALE = 1.0

# Phase 2: toggle + scale for the force/moment tracking compliance path (see
# _advance_force_tracking() below). Mirrors USE_STABILIZER_FEEDBACK/DELTA_SCALE's
# pattern: USE_FORCE_TRACKING=False is a strict no-op (feet[i]._compliance_coords
# never gets set, so _foot_to_coords() falls back to coords_ref unchanged), and
# FORCE_CTRL_SCALE=0.0 multiplies the final correction to exactly zero even when
# the path is enabled - both give the same "must match baseline" sanity check
# used earlier for DELTA_SCALE.
USE_FORCE_TRACKING = False
FORCE_CTRL_SCALE = 1.0

# The first FORCE_CTRL_SCALE sweep (0->1) at the placeholder base gains
# (force_ctrl_gain=moment_ctrl_gain=1e-5, see startSim()) showed almost no
# effect even at scale=1.0 (fell_at/forward_progress all within noise of the
# open-loop baseline) - those base gains are simply too small relative to the
# real force/moment errors (a few hundred N / tens of N*m) to produce a
# meaningful correction. FORCE_GAIN_MULTIPLIER scales the *base* gains
# themselves (see startSim()) by decades, independent of FORCE_CTRL_SCALE, so
# the two knobs can't be confused: this one searches for where an effect
# starts to appear at all; FORCE_CTRL_SCALE is for fine strength adjustment
# once a meaningful base magnitude is found.
#
# Sweep results (see run_trial cell): GAIN_MULT=1000 gave the largest forward-
# progress gain (~1.3-1.8x baseline) but also the highest max_tilt of any
# trial and was visually observed (interactive GUI run) to make the walk
# oscillate. Per that observation, defaulting to a weaker value that still
# showed a real forward-progress benefit in the long-run sweep (300: 0.341m
# vs baseline 0.254m) while keeping max_tilt at/below baseline (0.352 vs
# 0.354) - trading away some of the extra distance for a visibly smoother gait.
FORCE_GAIN_MULTIPLIER = 300.0


def _advance_com_state():
    """CoM/DCM integration, matching vnoid's Stabilizer.CalcDcmDynamicsSimple()
    with its defaults (no_dcm_gain=True, no_dcm_derivative=True):
    dcm_ref snaps directly to dcm_target (open-loop reference), and com_pos
    integrates toward dcm_ref with time-constant T. Feeding the lagged com_pos
    back into centroid.dcm_ref instead of dcm_target corrupts SteppingController's
    internal buffer/timing-adaptation state (stb0.dcm = centroid.dcm_ref.copy())
    and prevents forward progress from ever accumulating across steps."""
    global com_pos

    if com_pos is None:
        com_pos = centroid.com_pos_ref.copy()

    centroid.zmp_ref = centroid.zmp_target.copy()
    centroid.dcm_ref = centroid.dcm_target.copy()

    com_vel = (1.0 / param.T) * (centroid.dcm_ref - com_pos)
    com_pos = com_pos + com_vel * timer.dt
    centroid.com_pos_ref = com_pos.copy()


def _advance_com_state_stabilized():
    """Real-feedback counterpart to _advance_com_state(). Debugging history
    (three prior attempts, all diagnosed and ruled out in order):

    1. Called Stabilizer.CalcDcmDynamics() directly on the *shared* `centroid`
       object, letting centroid.dcm_ref drift persistently away from
       centroid.dcm_target (an integrated state there, not reset every tick).
       SteppingController.update() (new_stepping_controller.py) reads that same
       centroid.dcm_ref directly to predict the swing foot's landing position
       (`land_dcm`) and for timing adaptation, so any drift immediately became
       a lateral/forward offset in the next footstep's landing target - this
       is what the user observed as the swing foot stepping out to the side of
       the body on steps 1-2. Confirmed by logging dcm_ref drifting ~0.3m from
       dcm_target within a few hundred ticks.
    2. Ran CalcDcmDynamics() on an isolated "shadow" Centroid instead (so
       SteppingController's footstep planning was no longer touched - verified
       dcm_target/zmp_target logs became identical to the open-loop baseline).
       Still fell far earlier than baseline. Root cause: CalcDcmDynamics
       forward-Euler-integrates its own dcm_ref every tick, but
       SteppingController computes dcm_target from an *exact closed form*
       (`exp(t_ref/T)`, which grows up to ~70x within a single step duration
       given T~0.23s). At this pipeline's 100Hz control rate (dt=0.01, vs the
       1kHz/dt=0.001 the reference C++ design assumes), Euler-integrating that
       fast-growing exponential accumulates real numerical error versus the
       exact dcm_target trajectory - confirmed empirically: even with EVERY
       Stabilizer gain forced to exactly 0.0 (mathematically should collapse
       to the open-loop baseline), this shadow-integrator version still fell
       at iteration ~250, proving the divergence was a numerical-integration
       artifact, unrelated to any gain or sign in the recovery-moment math.

    Fix (this version): never let any dcm_ref persist/integrate across ticks
    at all - re-derive it fresh from dcm_target every tick, exactly like
    _advance_com_state(). This is mathematically exact (no Euler-integration
    error) because centroid.dcm_target is already recomputed in closed form
    every tick by SteppingController. Only the recovery-moment disturbance
    (Stabilizer.CalcRecoveryDelta(), a pure one-shot calculation with no
    persistent state) is added directly into this tick's CoM velocity - see
    DELTA_SCALE above for sweeping its strength during tuning."""
    global com_pos

    if com_pos is None:
        com_pos = centroid.com_pos_ref.copy()

    centroid.zmp_ref = centroid.zmp_target.copy()
    centroid.dcm_ref = centroid.dcm_target.copy()

    base_rpy = base.coords.RPY
    base_rpy_ref = base.coords_ref.RPY
    theta = np.array([base_rpy[0] - base_rpy_ref[0], base_rpy[1] - base_rpy_ref[1]])
    omega = np.array([base.angvel[0] - base.angvel_ref[0], base.angvel[1] - base.angvel_ref[1]])

    stabilizer.CalcBaseTilt(timer, param, base, theta, omega)
    delta = stabilizer.CalcRecoveryDelta(param, base, theta, omega) * DELTA_SCALE

    com_vel = (1.0 / param.T) * (centroid.dcm_ref - com_pos) + param.T * delta
    com_pos = com_pos + com_vel * timer.dt
    centroid.com_pos_ref = com_pos.copy()


def _advance_force_tracking():
    """Phase 2: force/moment tracking compliance, ported from the reference
    Update() loop's tail (stabilizer.py, array-based): computes each foot's
    desired ZMP/force/moment (Stabilizer.CalcForceDistribution(), using
    centroid.zmp_ref/force_ref/moment_ref and feet[i].contact_ref - the
    *planned* contact state SteppingController just set, not the measured one),
    then integrates a compliance offset (stabilizer.dpos/drot) from the error
    between that desired force/moment and the real measured foot.force/moment
    (from _update_foot_wrenches_from_sim()).

    Writes the corrected target into foot._compliance_coords (a fresh copy of
    foot.coords_ref, perturbed) rather than mutating foot.coords_ref itself:
    unlike centroid.dcm_ref, SteppingController fully *overwrites*
    foot[i].coords_ref from scratch every tick (no incremental integration), so
    there's no drift-corruption risk either way - but keeping the compliance
    correction in a separate attribute keeps this function's job (a pure
    add-on correction for _apply_full_body_ik() to consume) clearly separated
    from SteppingController's footstep-planning state.

    stabilizer.dpos/drot are the Stabilizer instance's own persistent
    integrators (reset to zero each startSim() since a fresh Stabilizer() is
    created there) - gains/limits/damping are set once in startSim() (see that
    docstring: the reference defaults are all 0, so those are placeholder
    starting points, not known-good values). FORCE_CTRL_SCALE multiplies only
    the *final* correction, so it can be swept from 0 upward independently of
    the underlying gains, mirroring DELTA_SCALE's role for the base-tilt path."""
    centroid.force_ref = np.array([0.0, 0.0, param.total_mass * param.gravity], dtype=np.float64)
    centroid.moment_ref = np.zeros(3, dtype=np.float64)

    stabilizer.CalcForceDistribution(param, centroid, feet)

    for i, f in enumerate(feet):
        f._compliance_coords = f.coords_ref.copy()
        if not f.contact:
            continue

        stabilizer.dpos[i] += (-stabilizer.force_ctrl_damping * stabilizer.dpos[i] +
                                stabilizer.force_ctrl_gain * (f.force_ref - f.force)) * timer.dt
        stabilizer.dpos[i] = np.clip(stabilizer.dpos[i], -stabilizer.force_ctrl_limit, stabilizer.force_ctrl_limit)

        stabilizer.drot[i] += (-stabilizer.moment_ctrl_damping * stabilizer.drot[i] +
                                stabilizer.moment_ctrl_gain * (f.moment_ref - f.moment)) * timer.dt
        stabilizer.drot[i] = np.clip(stabilizer.drot[i], -stabilizer.moment_ctrl_limit, stabilizer.moment_ctrl_limit)

        f._compliance_coords.pos = f._compliance_coords.pos - FORCE_CTRL_SCALE * stabilizer.dpos[i]
        rpy = f._compliance_coords.RPY - FORCE_CTRL_SCALE * stabilizer.drot[i]
        f._compliance_coords.setRPY(rpy)


def _apply_full_body_ik():
    """Solve the root pose and both legs simultaneously via
    ikrobot.legsCOM_IK() (irsl_choreonoid.robot_util.RobotModelWrapped
    .fullBodyIK()/legsCOM_IK()), replacing the previous pair of
    _apply_root_pose_to_robot() (root position only) + _apply_ik_targets()
    (each leg solved independently). Matches the pattern used in the vnoid
    reference sample (userdir/vnoid/vnoid_python/../walk_sim_vnoid.py::
    SimMain.step(), which calls `robot.fullBodyIK((rtgt, ltgt), (robot.rleg,
    robot.lleg), com_target=com_ref, ...)`): both feet and the CoM are
    constraints of a single IK solve, with the root free to translate in
    x/y/z (base_type='xyz') while its orientation is held at its current
    (upright) value - so balance is still governed purely by com_pos/DCM,
    never by directly commanding a base tilt here.

    com_target is com_pos + com_offset, not com_pos directly: legsCOM_IK()'s
    com_target drives the *true* mass-weighted center of mass, but com_pos
    throughout this pipeline is calibrated as root-link height (see startSim()
    where com_offset = true_CoM - root_pos is measured once at the standing
    pose - for HRP2 this is ~0.18m in Z, since the arms/chest/head pull the
    real CoM well above the waist). Using com_pos directly here commanded the
    real CoM down to root height, forcing an unstable deep crouch."""
    ltgt = _foot_to_coords(feet[0])
    rtgt = _foot_to_coords(feet[1])
    com_target = np.array(com_pos, dtype=np.float64) + com_offset
    try:
        ikrobot.legsCOM_IK(rtgt, ltgt, com_target=com_target,
                           com_constraint=[1, 1, 1],
                           base_type='xyz',
                           position_precision=[1e-9, 1e-9, 1e-9, 15e-9, 15e-9, 15e-9],
                           com_precision=[1e-9, 1e-9, 1e-9])
    except Exception:
        pass
    ikrobot.flush()


def _send_targets_to_simulation():
    """Push the kinematically-solved joint targets (root pose + leg IK, solved on
    the IK-only scratch model `ikrobot`) to the AIST physics simulation, following
    the same sequencer.setNoInterpolation()+runCount() pattern used by the proven
    working RL pipeline (irsl_rl/rl_env_cnoid.py::RLEnvChoreonoid.env_step()) and
    the vnoid walking demo (userdir/humanoid_research_k/humanoid_research/
    walk_sim_vnoid.py::SimMain.step()): hold this tick's target for `substeps`
    physics ticks while the PD controller (gravity + contact forces included)
    drives the simulated body toward it."""
    target_av = ikrobot.angleVector()
    sim.sequencer.setNoInterpolation([target_av], env_cfg['substeps'])
    sim.runCount(env_cfg['substeps'], update=None, stop=False)


def step():
    global time, current_step, step_progress

    if stepping_controller is None:
        raise RuntimeError('call startSim() before step()')

    timer.time = time
    if not hasattr(timer, 'count'):
        timer.count = 0
    timer.count += 1

    # Sense: Robot/RobotMujoco::Sense
    _sync_robot_state_from_sim()
    _update_foot_wrenches_from_sim()
    _update_measured_zmp()

    # SteppingController::Update (this now also updates base.coords_ref's yaw
    # to the midpoint of the two feet internally, so no separate step is needed
    # here for that, unlike the old array-based SteppingController usage).
    active = stepping_controller.update(
        timer,
        param,
        footstep,
        footstep_buffer,
        centroid,
        base,
        feet,
    )

    if not active:
        return False

    # Stabilizer::Update equivalent: DCM/CoM feedback integration. Toggle
    # between the pure open-loop baseline and the real-feedback (base tilt)
    # path via USE_STABILIZER_FEEDBACK.
    if USE_STABILIZER_FEEDBACK:
        _advance_com_state_stabilized()
    else:
        _advance_com_state()

    # Phase 2: force/moment tracking compliance (see _advance_force_tracking()
    # docstring). Runs after SteppingController has set this tick's
    # feet[i].coords_ref/contact_ref, and before _apply_full_body_ik() reads
    # the (possibly compliance-corrected) foot targets via _foot_to_coords().
    if USE_FORCE_TRACKING:
        _advance_force_tracking()

    # IK: solve root pose + both legs simultaneously against the target CoM
    # position and foot placements (see _apply_full_body_ik() docstring for
    # why this replaced the earlier independent per-leg IK + root-set pair).
    _apply_full_body_ik()

    # Drive the physics simulation toward those targets (PD control + gravity + contacts).
    _send_targets_to_simulation()

    time += timer.dt
    current_step = len(footstep.steps)
    step_progress = min(1.0, time / max(env_cfg['episode_length_s'], 1.0))
    return True

In [27]:
def run_trial(label, use_stabilizer, max_iterations=2200, delta_scale=1.0, use_force_tracking=False, force_ctrl_scale=1.0, force_gain_multiplier=1.0):
    """Run one full walking trial from a fresh startSim() and return summary
    metrics. use_stabilizer selects _advance_com_state() (open-loop baseline)
    vs _advance_com_state_stabilized() (measured base-tilt feedback) via the
    module-level USE_STABILIZER_FEEDBACK flag (see step() in the previous cell).
    delta_scale sets the module-level DELTA_SCALE (recovery-moment strength;
    only relevant when use_stabilizer=True). use_force_tracking/force_ctrl_scale
    similarly control the Phase 2 force/moment tracking compliance path
    (USE_FORCE_TRACKING/FORCE_CTRL_SCALE, see _advance_force_tracking()).
    force_gain_multiplier sets FORCE_GAIN_MULTIPLIER, which scales the base
    force_ctrl_gain/moment_ctrl_gain themselves (set in startSim()) by decades -
    distinct from force_ctrl_scale, which only scales the final correction."""
    global USE_STABILIZER_FEEDBACK, DELTA_SCALE, USE_FORCE_TRACKING, FORCE_CTRL_SCALE, FORCE_GAIN_MULTIPLIER

    USE_STABILIZER_FEEDBACK = use_stabilizer
    DELTA_SCALE = delta_scale
    USE_FORCE_TRACKING = use_force_tracking
    FORCE_CTRL_SCALE = force_ctrl_scale
    FORCE_GAIN_MULTIPLIER = force_gain_multiplier
    startSim()
    print('--- trial: {} (USE_FORCE_TRACKING={}, FORCE_CTRL_SCALE={}, FORCE_GAIN_MULTIPLIER={}) ---'.format(
        label, use_force_tracking, force_ctrl_scale, force_gain_multiplier))
    print('com_height used for walking plan (actual settled height after 5s):', param.com_height)

    fall_limit_rad = np.deg2rad(min(env_cfg['termination_if_roll_greater_than'], env_cfg['termination_if_pitch_greater_than']))

    base_heights = []
    base_tilts = []
    base_xs = []
    fell_at = None
    for i in range(max_iterations):
        if not step():
            print('Walking simulation finished at iteration {}'.format(i))
            break
        base_pos = base.coords.pos
        base_rpy = base.coords.RPY
        base_heights.append(base_pos[2])
        base_xs.append(base_pos[0])
        tilt = max(abs(base_rpy[0]), abs(base_rpy[1]))
        base_tilts.append(tilt)
        if tilt > fall_limit_rad:
            print('Robot fell over at iteration {} (tilt={:.3f} rad > {:.3f} rad limit)'.format(i, tilt, fall_limit_rad))
            fell_at = i
            break
    else:
        print('Walking simulation reached iteration limit without finishing')

    if base_heights:
        print('base height [m]: min={:.3f} max={:.3f} final={:.3f}'.format(
            min(base_heights), max(base_heights), base_heights[-1]))
    if base_xs:
        print('base x [m]: start={:.3f} final={:.3f} (forward progress={:.3f})'.format(
            base_xs[0], base_xs[-1], base_xs[-1] - base_xs[0]))
    if base_tilts:
        print('max |roll|/|pitch| [rad]: {:.3f} (limit ~{:.3f})'.format(
            max(base_tilts), fall_limit_rad))

    return {
        'label': label,
        'fell_at': fell_at,
        'iterations': len(base_xs),
        'forward_progress': (base_xs[-1] - base_xs[0]) if base_xs else 0.0,
        'max_tilt': max(base_tilts) if base_tilts else 0.0,
        'fall_limit_rad': fall_limit_rad,
        'height_min': min(base_heights) if base_heights else None,
        'height_max': max(base_heights) if base_heights else None,
    }


# Phase 2 gain-magnitude sweep: the earlier FORCE_CTRL_SCALE sweep (0->1) at
# the placeholder base gains (1e-5) showed essentially no effect at all - so
# instead of the final-correction multiplier, sweep the *base* gain magnitude
# itself (FORCE_GAIN_MULTIPLIER, applied on top of the 1e-5 base in startSim())
# by decades, with FORCE_CTRL_SCALE pinned at 1.0, to find where a real effect
# (positive or negative) starts to appear.
results = []
results.append(run_trial('open-loop (baseline)', use_stabilizer=False, max_iterations=260))
for mult in (1, 10, 100, 1000, 10000):
    results.append(run_trial('force tracking (GAIN_MULT={})'.format(mult),
                              use_stabilizer=False, max_iterations=260,
                              use_force_tracking=True, force_ctrl_scale=1.0,
                              force_gain_multiplier=mult))

print()
print('=== Phase 2 gain-magnitude sweep ===')
print('{:<40s} {:>10s} {:>12s} {:>10s}'.format('trial', 'fell_at', 'forward[m]', 'max_tilt'))
for r in results:
    print('{:<40s} {:>10s} {:>12.3f} {:>10.3f}'.format(
        r['label'], str(r['fell_at']), r['forward_progress'], r['max_tilt']))

--- trial: open-loop (baseline) (USE_FORCE_TRACKING=False, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=1.0) ---
com_height used for walking plan (actual settled height after 5s): 0.5274478915405062
Robot fell over at iteration 232 (tilt=0.352 rad > 0.349 rad limit)
base height [m]: min=0.516 max=0.530 final=0.523
base x [m]: start=-0.030 final=0.227 (forward progress=0.257)
max |roll|/|pitch| [rad]: 0.352 (limit ~0.349)
--- trial: force tracking (GAIN_MULT=1) (USE_FORCE_TRACKING=True, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=1) ---
com_height used for walking plan (actual settled height after 5s): 0.5276220638754957
Robot fell over at iteration 236 (tilt=0.351 rad > 0.349 rad limit)
base height [m]: min=0.518 max=0.531 final=0.524
base x [m]: start=-0.029 final=0.223 (forward progress=0.252)
max |roll|/|pitch| [rad]: 0.351 (limit ~0.349)
--- trial: force tracking (GAIN_MULT=10) (USE_FORCE_TRACKING=True, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=10) ---
com_height used for walking 

In [28]:
# Phase 2 validation: sanity-check the real foot force/torque sensors and the
# resulting Stabilizer.CalcZmp() output before wiring anything into control.
# Runs the open-loop baseline (USE_STABILIZER_FEEDBACK=False - the measured-ZMP
# path is purely observational right now, see _update_measured_zmp()) and
# prints feet[].force/moment/contact/balance/zmp plus centroid.zmp at a few
# points: right after standing settles (double support, expect force[2] on
# each foot roughly totaling body weight/2 each) and periodically during
# walking (expect contact to alternate and balance to shift toward the
# support foot). Capped at 220 iterations (below the ~230-250 fall point seen
# in Phase 1 verification) so a fall doesn't trigger a slow IK-failure retry
# storm that blows past the notebook execution timeout.
USE_STABILIZER_FEEDBACK = False
DELTA_SCALE = 1.0
startSim()

print('total_mass*g ~= {:.1f} N (expect roughly this much total vertical force across both feet)'.format(param.total_mass * 9.8))

for i in range(220):
    if not step():
        print('Walking simulation finished at iteration {}'.format(i))
        break
    if i < 5 or i % 100 == 0:
        print('--- iter {} ---'.format(i))
        for name, f in zip(('lleg', 'rleg'), feet):
            print('  {}: force={} moment={} contact={} balance={}'.format(
                name, f.force, f.moment, getattr(f, 'contact', None), getattr(f, 'balance', None)))
        print('  centroid.zmp = {}  (centroid.zmp_target = {})'.format(centroid.zmp, centroid.zmp_target))

total_mass*g ~= 550.5 N (expect roughly this much total vertical force across both feet)
--- iter 0 ---
  lleg: force=[ 1.75761872e-01 -1.30288694e+01  2.59027786e+02] moment=[1.03402549 3.5152693  0.70737161] contact=True balance=0.4993887374787488
  rleg: force=[  0.26230213  12.97824258 259.66189729] moment=[-0.89664682  3.22845961 -0.70885895] contact=True balance=0.5006112625212511
  centroid.zmp = [-0.01300147  0.00011204  0.        ]  (centroid.zmp_target = [-1.08911211e-03  1.65634042e-03 -1.11022302e-16])
--- iter 1 ---
  lleg: force=[ -93.94187541 -208.2893346   534.55501235] moment=[43.34842918 -7.51314806 -5.15843827] contact=True balance=0.33109011864318727
  rleg: force=[ 467.40413737  417.36754056 1079.97523862] moment=[-93.01801519 127.47745422 -16.2134183 ] contact=True balance=0.6689098813568127
  centroid.zmp = [-0.07430292 -0.07299158  0.        ]  (centroid.zmp_target = [-1.08911211e-03  1.65634042e-03 -1.11022302e-16])
--- iter 2 ---
  lleg: force=[59.34377859  2.

In [29]:
# Follow-up to the GAIN_MULT sweep above: GAIN_MULT=100 uniquely survived the
# full 260-iteration cap (vs. baseline falling at 258) but with forward
# progress collapsing to 0.067m (baseline 0.238m) - i.e. a real but very
# conservative/damping effect. GAIN_MULT=1000 gave the best forward progress
# (0.358m) of any trial but still fell around iter 225. This cell runs a
# longer window (1200 iterations) to see how far GAIN_MULT=100 actually gets
# before it (if ever) falls, and narrows the search between 100 and 1000 to
# look for a value that keeps the no-fall behavior while recovering more
# forward progress.
results_longrun = []
results_longrun.append(run_trial('open-loop (baseline, long)', use_stabilizer=False, max_iterations=1200))
for mult in (100, 200, 300, 500, 1000):
    results_longrun.append(run_trial('force tracking (GAIN_MULT={}, long)'.format(mult),
                                      use_stabilizer=False, max_iterations=1200,
                                      use_force_tracking=True, force_ctrl_scale=1.0,
                                      force_gain_multiplier=mult))

print()
print('=== Phase 2 gain-magnitude long-run sweep ===')
print('{:<40s} {:>10s} {:>12s} {:>10s}'.format('trial', 'fell_at', 'forward[m]', 'max_tilt'))
for r in results_longrun:
    print('{:<40s} {:>10s} {:>12.3f} {:>10.3f}'.format(
        r['label'], str(r['fell_at']), r['forward_progress'], r['max_tilt']))


--- trial: open-loop (baseline, long) (USE_FORCE_TRACKING=False, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=1.0) ---
com_height used for walking plan (actual settled height after 5s): 0.527454694061947
Robot fell over at iteration 247 (tilt=0.352 rad > 0.349 rad limit)
base height [m]: min=0.515 max=0.532 final=0.519
base x [m]: start=-0.030 final=0.208 (forward progress=0.238)
max |roll|/|pitch| [rad]: 0.352 (limit ~0.349)
--- trial: force tracking (GAIN_MULT=100, long) (USE_FORCE_TRACKING=True, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=100) ---
com_height used for walking plan (actual settled height after 5s): 0.527477201261437
Robot fell over at iteration 234 (tilt=0.354 rad > 0.349 rad limit)
base height [m]: min=0.514 max=0.560 final=0.514
base x [m]: start=-0.031 final=0.247 (forward progress=0.278)
max |roll|/|pitch| [rad]: 0.354 (limit ~0.349)
--- trial: force tracking (GAIN_MULT=200, long) (USE_FORCE_TRACKING=True, FORCE_CTRL_SCALE=1.0, FORCE_GAIN_MULTIPLIER=200) ---
com_he